# 개별종목 조합H — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합H 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합H의 피처 값만 지정합니다.
import json

COMBINATION = 'H'
FEATURE_COLUMNS = (
    'dist_high_60',
    'sma_gap_20_60',
    'relative_ret_5_market',
    'rsi_14',
    'hv_regime',
    'turnover_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합H 피처: ('dist_high_60', 'sma_gap_20_60', 'relative_ret_5_market', 'rsi_14', 'hv_regime', 'turnover_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4647,0.5012,-0.0365,0.3541,0.3672,0.0690,0.3815,0.1801,0.2849
1,2,balanced,980,20150123,20150421,0.3763,0.3978,-0.0215,0.3384,0.3536,0.0369,0.3647,0.2558,0.3151
2,3,balanced,1210,20151228,20160328,0.3720,0.3762,-0.0041,0.3470,0.3573,0.0416,0.3556,0.2406,0.3085
3,4,balanced,1439,20161202,20170228,0.4723,0.4617,0.0105,0.4048,0.4107,0.1364,0.4171,0.2181,0.3271
4,5,balanced,1669,20171113,20180207,0.4224,0.3901,0.0324,0.3962,0.4029,0.1125,0.4019,0.3476,0.3862
5,6,balanced,1899,20181024,20190118,0.4200,0.3725,0.0476,0.4148,0.4148,0.1250,0.4275,0.3543,0.3940
6,7,balanced,2129,20190930,20191224,0.4450,0.4781,-0.0331,0.3760,0.3836,0.0911,0.3974,0.2789,0.3533
7,8,balanced,2359,20200902,20201130,0.3721,0.3476,0.0245,0.3692,0.3817,0.0719,0.3864,0.5074,0.4072
8,9,balanced,2589,20210806,20211105,0.3602,0.3914,-0.0312,0.3574,0.3695,0.0495,0.3690,0.3175,0.3439
9,10,balanced,2818,20220714,20221012,0.3782,0.3454,0.0328,0.3756,0.3831,0.0738,0.3766,0.2851,0.3403


,OOS 폴드 평균
accuracy,0.4048
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0080
macro_f1,0.3749
balanced_accuracy,0.3826
mcc,0.0802
pr_auc_macro_ovr,0.3883
down_recall,0.3143
core_harmonic_mean,0.3529


재실행 명령: python scripts/run_stock_model_experiment.py
